In [15]:
import logging

import bm25s

from beir import LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch

In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

## Data Loading

In [3]:
BEIR_DATA_ROOT = "../data/beir"

In [8]:
corpus, queries, qrels = GenericDataLoader(
    data_folder=BEIR_DATA_ROOT, qrels_file=f"{BEIR_DATA_ROOT}/qrels.tsv"
).load_custom()

2024-12-28 16:33:19 - Loading Corpus...


100%|██████████| 1582/1582 [00:00<00:00, 153757.13it/s]

2024-12-28 16:33:19 - Loaded 1582 Documents.
2024-12-28 16:33:19 - Doc Example: {'text': 'bogor hotél institut (bhi) gawé bareng jeung forum indonesé-nederland (fined), sarta hotél salak the heritage gel "holland indonésé festival" di hotél salak, jalan ir jonda, kota bogor, (15/11/2009). festival anu buka wakil duta besar walanda keur indonésé, annemieke ruigrok boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat. wakil duta besar walanda, annemieke ruigrok kaku pohara gumbira ku lumangsungna acara ieu. “indonésé jeung walanda boga hubungan lit,“ ceuk manéhna. "sanajan walanda boga sajarah mangsa ka tukang anu kurang alus di mata masarakat indonésé, tapi henteu ngaleungitkeun hubungan eta," sambungna. lamun tempo sajarah mangsa ka tukang indonésé jeung walanda boga sawatara kamiripan dina hal kabudayan. teu eutik warga walanda anu mikaresep seni budaya anu asalna ti indonésé. "urang walanda pohara resep ku masak urang indonésé," terus dina basa walanda. "ku

## Evaluate using Sentence Transformers

In [5]:
model = DenseRetrievalExactSearch(
    models.SentenceBERT("msmarco-distilbert-base-tas-b"), batch_size=16
)
model

2024-12-28 16:22:46 - Load pretrained SentenceTransformer: msmarco-distilbert-base-tas-b


/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)


2024-12-28 16:30:16 - Use pytorch device: cuda


In [13]:
retriever = EvaluateRetrieval(
    model, score_function="cos_sim"
)  # or "cos_sim" for cosine similarity
results = retriever.retrieve(corpus, queries)

2024-12-28 16:34:24 - Encoding Queries...


Batches: 100%|██████████| 495/495 [00:04<00:00, 100.79it/s]


2024-12-28 16:34:29 - Sorting Corpus by document length (Longest first)...
2024-12-28 16:34:29 - Encoding Corpus in batches... Warning: This might take a while!
2024-12-28 16:34:29 - Scoring Function: Cosine Similarity (cos_sim)
2024-12-28 16:34:29 - Encoding Batch 1/1...


Batches: 100%|██████████| 99/99 [00:07<00:00, 12.69it/s]


In [14]:
#### Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K  where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)

2024-12-28 16:34:42 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2024-12-28 16:34:45 - 

2024-12-28 16:34:45 - NDCG@1: 0.0976
2024-12-28 16:34:45 - NDCG@3: 0.1336
2024-12-28 16:34:45 - NDCG@5: 0.1467
2024-12-28 16:34:45 - NDCG@10: 0.1645
2024-12-28 16:34:45 - NDCG@100: 0.2135
2024-12-28 16:34:45 - NDCG@1000: 0.2645
2024-12-28 16:34:45 - 

2024-12-28 16:34:45 - MAP@1: 0.0976
2024-12-28 16:34:45 - MAP@3: 0.1246
2024-12-28 16:34:45 - MAP@5: 0.1318
2024-12-28 16:34:45 - MAP@10: 0.1391
2024-12-28 16:34:45 - MAP@100: 0.1475
2024-12-28 16:34:45 - MAP@1000: 0.1491
2024-12-28 16:34:45 - 

2024-12-28 16:34:45 - Recall@1: 0.0976
2024-12-28 16:34:45 - Recall@3: 0.1599
2024-12-28 16:34:45 - Recall@5: 0.1918
2024-12-28 16:34:45 - Recall@10: 0.2469
2024-12-28 16:34:45 - Recall@100: 0.4953
2024-12-28 16:34:45 - Recall@1000: 0.9202
2024-12-28 16:34:45 - 

2024-12-28 16:34:45 - P@1: 0.0976
2024-12-28 16:34:45

## Evaluate using BM25

Source: https://www.kaggle.com/code/xhlulu/benchmark-bm25-on-beir

In [25]:
def postprocess_results_for_eval(results, scores, query_ids):
    """
    Given the queried results and scores output by BM25S, postprocess them
    to be compatible with BEIR evaluation functions.
    query_ids is a list of query ids in the same order as the results.
    """

    results_record = [
        {"id": qid, "hits": results[i], "scores": list(scores[i])}
        for i, qid in enumerate(query_ids)
    ]

    result_dict_for_eval = {
        res["id"]: {
            docid: float(score) for docid, score in zip(res["hits"], res["scores"])
        }
        for res in results_record
    }

    return result_dict_for_eval

In [16]:
corpus_ids, corpus_lst = [], []
for key, val in corpus.items():
    corpus_ids.append(key)
    corpus_lst.append(val["title"] + " " + val["text"])

qids, queries_lst = [], []
for key, val in queries.items():
    qids.append(key)
    queries_lst.append(val)

In [22]:
corpus_tokens = bm25s.tokenize(corpus_lst)
query_tokens = bm25s.tokenize(queries_lst)

In [23]:
model = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
model.index(corpus_tokens)

2024-12-28 16:48:09 - Building index from IDs objects


In [24]:
queried_results, queried_scores = model.retrieve(
    query_tokens, corpus=corpus_ids, k=1000, n_threads=1
)

In [26]:
results_dict = postprocess_results_for_eval(queried_results, queried_scores, qids)

In [28]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, results_dict, [1, 3, 5, 10, 100, 1000]
)

2024-12-28 16:50:23 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - NDCG@1: 0.5321
2024-12-28 16:50:26 - NDCG@3: 0.6313
2024-12-28 16:50:26 - NDCG@5: 0.6512
2024-12-28 16:50:26 - NDCG@10: 0.6704
2024-12-28 16:50:26 - NDCG@100: 0.6951
2024-12-28 16:50:26 - NDCG@1000: 0.7023
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - MAP@1: 0.5321
2024-12-28 16:50:26 - MAP@3: 0.6078
2024-12-28 16:50:26 - MAP@5: 0.6189
2024-12-28 16:50:26 - MAP@10: 0.6269
2024-12-28 16:50:26 - MAP@100: 0.6320
2024-12-28 16:50:26 - MAP@1000: 0.6322
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - Recall@1: 0.5321
2024-12-28 16:50:26 - Recall@3: 0.6990
2024-12-28 16:50:26 - Recall@5: 0.7470
2024-12-28 16:50:26 - Recall@10: 0.8061
2024-12-28 16:50:26 - Recall@100: 0.9219
2024-12-28 16:50:26 - Recall@1000: 0.9805
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - P@1: 0.5321
2024-12-28 16:50:26